# 🦜 Colombian Bird Song — Audio Pre-Processing Pipeline

This notebook prepares the audio dataset for model training. It fetches bird recording metadata directly from [Xeno-Canto](https://xeno-canto.org), filters recordings/species using your project rules, downloads MP3 files to Google Drive, and prepares the next steps for mel-spectrogram conversion.

---

## Pipeline Overview

| Step | Description | Status |
|------|-------------|--------|
| 1 | Setup & Authentication | ✅ |
| 2 | Mount Drive & Configure Paths | ✅ |
| 3 | Fetch Metadata from Xeno-Canto | ✅ |
| 4 | Filter Species & Recordings | ✅ |
| 5 | Download Recordings to Drive | ✅ |
| 6 | Convert audio → mel-spectrograms | 🔜 |
| 7 | Export tensors for training | 🔜 |

## Step 1 — Setup & Authentication

Install required packages and enter your Xeno-Canto API key securely in the runtime.

Generate the key from your Xeno-Canto account page: https://xeno-canto.org/account/api

The notebook cannot access your account or generate the key for you.

In [ ]:
# Install required packages
%pip install requests

print("Enter your Xeno-Canto API key.")
print("Generate it from your account page: https://xeno-canto.org/account/api")
print("This notebook cannot access your account or create a key for you.")
XC_API_KEY = input("XC_API_KEY: ").strip()

if not XC_API_KEY:
    print("Warning: no API key provided. Metadata fetching will fail until one is set.")

## Step 2 — Mount Drive & Configure Paths

Mount Google Drive for audio output and configure a temporary runtime folder for intermediate JSON metadata.

- Metadata files are stored in `/content/data` (temporary, cleared after session).
- Audio files are stored in your Drive output folder.

In [ ]:
from pathlib import Path
from google.colab import drive

# Mount Drive for persistent audio storage
drive.mount('/content/drive')

DRIVE_ROOT_PATH = "/content/drive/MyDrive"
DOWNLOAD_FOLDER_NAME = "bird_songs"

TEMP_DATA_DIR = Path("/content/data")
RAW_DIR = TEMP_DATA_DIR / "raw"
RAW_JSON_PATH = TEMP_DATA_DIR / "colombia_birds_by_species.json"
FILTERED_JSON_PATH = TEMP_DATA_DIR / "colombia_birds_filtered.json"

RAW_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = Path(DRIVE_ROOT_PATH) / DOWNLOAD_FOLDER_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Temporary metadata dir: {TEMP_DATA_DIR}")
print(f"Raw page dir: {RAW_DIR}")
print(f"Drive output dir: {OUTPUT_DIR}")

## Step 3 — Fetch Metadata from Xeno-Canto

Fetch all Colombian bird recordings from the Xeno-Canto API v3 (`cnt:colombia grp:birds`), save raw page responses, validate country/group/coordinates, then aggregate results by species.

In [ ]:
import json
import requests

BASE_URL = "https://xeno-canto.org/api/3/recordings"
QUERY = "cnt:colombia grp:birds"
PER_PAGE = 500

COL_LAT_MIN, COL_LAT_MAX = -4.5, 13.0
COL_LNG_MIN, COL_LNG_MAX = -82.0, -66.5


def fetch_page(query: str, page: int) -> dict:
    params = {
        "query": query,
        "key": XC_API_KEY,
        "per_page": PER_PAGE,
        "page": page,
    }
    response = requests.get(
        BASE_URL,
        params=params,
        headers={"Accept": "application/json"},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


def save_raw_page(data: dict, page: int) -> None:
    with open(RAW_DIR / f"page_{page:04d}.json", "w", encoding="utf-8") as fh:
        json.dump(data, fh, ensure_ascii=False, indent=2)


def in_colombia_bbox(lat: str, lon: str) -> bool:
    try:
        flat, flon = float(lat), float(lon)
    except (ValueError, TypeError):
        return True
    return (COL_LAT_MIN <= flat <= COL_LAT_MAX) and (COL_LNG_MIN <= flon <= COL_LNG_MAX)


def validate_colombian_birds(recordings: list[dict]) -> list[dict]:
    validated = []
    for r in recordings:
        if r.get("grp", "").lower() != "birds":
            continue
        if r.get("cnt", "").lower() != "colombia":
            continue
        if not in_colombia_bbox(r.get("lat", ""), r.get("lon", "")):
            continue
        validated.append(r)
    return validated


def group_by_species(recordings: list[dict]) -> dict:
    species_map = {}

    for r in recordings:
        genus = r.get("gen", "").strip()
        epithet = r.get("sp", "").strip()
        key = f"{genus} {epithet}".strip()

        if key not in species_map:
            species_map[key] = {
                "scientific_name": key,
                "genus": genus,
                "species": epithet,
                "subspecies_seen": sorted({r.get("ssp", "").strip()} - {""}),
                "english_name": r.get("en", ""),
                "group": r.get("grp", ""),
                "recording_count": 0,
                "recordings": [],
            }

        entry = species_map[key]

        ssp = r.get("ssp", "").strip()
        if ssp and ssp not in entry["subspecies_seen"]:
            entry["subspecies_seen"].append(ssp)

        entry["recording_count"] += 1
        entry["recordings"].append(
            {
                "id": r.get("id", ""),
                "quality": r.get("q", ""),
                "length": r.get("length", ""),
                "file": r.get("file", ""),
                "file_name": r.get("file-name", ""),
                "type": r.get("type", ""),
                "sex": r.get("sex", ""),
                "stage": r.get("stage", ""),
                "method": r.get("method", ""),
                "location": r.get("loc", ""),
                "lat": r.get("lat", ""),
                "lon": r.get("lon", ""),
                "date": r.get("date", ""),
                "recordist": r.get("rec", ""),
                "license": r.get("lic", ""),
                "xc_url": r.get("url", ""),
                "also": r.get("also", []),
                "animal_seen": r.get("animal-seen", ""),
                "remarks": r.get("rmk", ""),
            }
        )

    return species_map


def fetch_all_pages() -> list[dict]:
    page = 1
    print(f"[page {page}] fetching ...")
    data = fetch_page(QUERY, page)

    num_pages = int(data.get("numPages", 1))
    print(f"Total pages: {num_pages}")

    all_recordings = list(data.get("recordings", []))
    save_raw_page(data, page)

    for page in range(2, num_pages + 1):
        print(f"[page {page}/{num_pages}] fetching ...")
        data = fetch_page(QUERY, page)
        recordings = data.get("recordings", [])
        all_recordings.extend(recordings)
        save_raw_page(data, page)

    return all_recordings


print("=" * 60)
print("Xeno-Canto metadata fetch")
print("=" * 60)

if not XC_API_KEY:
    print("No XC_API_KEY set. Run Step 1 again and provide your key.")
else:
    all_recordings = fetch_all_pages()
    print(f"Fetched total recordings: {len(all_recordings)}")

    validated = validate_colombian_birds(all_recordings)
    print(f"Validated recordings: {len(validated)}")
    print(f"Discarded recordings: {len(all_recordings) - len(validated)}")

    species_map = group_by_species(validated)

    output = {
        "query": QUERY,
        "total_recordings": len(validated),
        "total_species": len(species_map),
        "species": dict(sorted(species_map.items())),
    }

    with open(RAW_JSON_PATH, "w", encoding="utf-8") as fh:
        json.dump(output, fh, ensure_ascii=False, indent=2)

    print(f"Grouped metadata saved to: {RAW_JSON_PATH}")
    print(f"Raw API page files saved under: {RAW_DIR}")

## Step 4 — Filter Species & Recordings

Apply project filtering rules to the grouped metadata:

- Keep only quality `A` or `B`
- Keep durations between `3` and `60` seconds
- Keep only species with more than `35` surviving recordings

The filtered output is saved to `/content/data/colombia_birds_filtered.json`.

In [ ]:
import json

MIN_LENGTH_SEC = 3
MAX_LENGTH_SEC = 60
ACCEPTABLE_QUALITY = frozenset({"A", "B"})
MIN_SPECIES_RECORDINGS = 35


def length_seconds(s: str) -> int:
    parts = [int(p) for p in s.strip().split(":")]
    if len(parts) == 2:
        return parts[0] * 60 + parts[1]
    if len(parts) == 3:
        return parts[0] * 3600 + parts[1] * 60 + parts[2]
    raise ValueError(f"Unexpected length format: {s!r}")


if not RAW_JSON_PATH.exists():
    print(f"Missing grouped metadata: {RAW_JSON_PATH}")
else:
    with open(RAW_JSON_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    if "species" not in data:
        raise KeyError("Input JSON is missing the 'species' key.")

    new_species = {}
    total_recordings = 0

    for name, info in data["species"].items():
        if info.get("recording_count", 0) < MIN_SPECIES_RECORDINGS:
            continue

        kept = []
        for r in info["recordings"]:
            if not r.get("length"):
                continue

            q = (r.get("quality") or "").strip()
            if q not in ACCEPTABLE_QUALITY:
                continue

            duration = length_seconds(r["length"])
            if not (MIN_LENGTH_SEC <= duration <= MAX_LENGTH_SEC):
                continue

            kept.append(r)

        if len(kept) <= MIN_SPECIES_RECORDINGS:
            continue

        species_out = dict(info)
        species_out["recordings"] = kept
        species_out["recording_count"] = len(kept)
        new_species[name] = species_out
        total_recordings += len(kept)

    for name, species_out in new_species.items():
        assert species_out["recording_count"] > MIN_SPECIES_RECORDINGS, (
            f"Integrity check failed for {name!r}: recording_count={species_out['recording_count']}"
        )

    out_data = {
        "query": data["query"],
        "total_recordings": total_recordings,
        "total_species": len(new_species),
        "species": new_species,
    }

    with open(FILTERED_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(out_data, f, ensure_ascii=False, indent=2)

    print(f"Filtered species: {len(new_species)}")
    print(f"Filtered recordings: {total_recordings}")
    print(f"Filtered metadata saved to: {FILTERED_JSON_PATH}")

## Step 5 — Download Recordings to Google Drive

Load filtered metadata from the temporary runtime path and download MP3 files to Drive, grouped by species. Existing files are skipped automatically.

In [ ]:
import json
import requests
import concurrent.futures
from pathlib import Path

if not FILTERED_JSON_PATH.exists():
    print(f"Cannot find filtered metadata at: {FILTERED_JSON_PATH}")
else:
    with open(FILTERED_JSON_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(
        f"Loaded metadata for {data.get('total_species', 0)} species and "
        f"{data.get('total_recordings', 0)} recordings."
    )

    download_tasks = []

    for species_name, species_info in data.get("species", {}).items():
        safe_species_name = species_info.get("scientific_name", species_name).replace(" ", "_")
        species_dir = OUTPUT_DIR / safe_species_name
        species_dir.mkdir(parents=True, exist_ok=True)

        for recording in species_info.get("recordings", []):
            file_url = recording.get("file")
            rec_id = recording.get("id")
            if file_url and rec_id:
                out_file = species_dir / f"XC{rec_id}.mp3"
                download_tasks.append((file_url, out_file))

    print(f"Prepared {len(download_tasks)} files for download to {OUTPUT_DIR}")

    def download_file(url: str, out_path: Path) -> bool:
        if out_path.exists():
            return False
        try:
            if url.startswith("//"):
                url = "https:" + url

            response = requests.get(url, timeout=30)
            response.raise_for_status()
            with open(out_path, "wb") as f_out:
                f_out.write(response.content)
            return True
        except Exception:
            return False

    MAX_WORKERS = 4
    downloaded_count = 0
    skipped_count = 0

    print("Starting downloads...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_item = {
            executor.submit(download_file, url, path): (url, path)
            for url, path in download_tasks
        }

        for index, future in enumerate(concurrent.futures.as_completed(future_to_item), start=1):
            if future.result():
                downloaded_count += 1
            else:
                skipped_count += 1

            if index % 100 == 0:
                print(f"Processed {index}/{len(download_tasks)} items...")

    print(f"Download complete. Downloaded: {downloaded_count}, skipped: {skipped_count}")

## Step 6 — Convert Audio to Mel-Spectrograms

> 🔜 **Coming soon.** This step will load each MP3 from Drive, compute mel-spectrogram tensors, and save them for model training.

## Step 7 — Export Tensors for Training

> 🔜 **Coming soon.** This step will package spectrogram tensors and labels into training-ready datasets (for example, TensorFlow or PyTorch pipelines).